In [0]:
%sql
DROP TABLE IF EXISTS azuresalesdatabricks.gold.dim_category;

In [0]:
dbutils.fs.rm("abfss://gold@azuresales.dfs.core.windows.net/_checkpoints/dim_category/", recurse=True)

# 1. Create the gold schema and managed location in Unity Catalog

In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS azuresalesdatabricks.gold
MANAGED LOCATION 'abfss://gold@azuresales.dfs.core.windows.net/';

# 2. dim_date

## 2.1. Create the table — the one gold table that doesn't come from any of our source data at all, since it's a pure calendar.

In [0]:
from pyspark.sql import functions as F

# ============================================================
# dim_date has no source table — we're GENERATING it from a
# date range, wide enough to safely cover our fact data
# (2024-01-15 to 2024-01-21) plus buffer for any future data.
# ============================================================
start_date = "2020-01-01"
end_date   = "2030-12-31"

# sequence() builds an array of every date in the range;
# explode() turns that one array into one row per date
date_df = (spark.sql(f"SELECT explode(sequence(to_date('{start_date}'), to_date('{end_date}'), interval 1 day)) AS full_date"))

dim_date = (date_df
    # date_key: surrogate key in YYYYMMDD format — a common
    # convention because it's both unique AND sorts naturally
    .withColumn("date_key", F.date_format("full_date", "yyyyMMdd"))
    .withColumn("day", F.dayofmonth("full_date"))
    .withColumn("month", F.month("full_date"))
    .withColumn("month_name", F.date_format("full_date", "MMMM"))
    .withColumn("quarter", F.quarter("full_date"))
    .withColumn("year", F.year("full_date"))
    .withColumn("day_of_week", F.date_format("full_date", "EEEE"))
    # dayofweek() in Spark returns 1=Sunday...7=Saturday
    .withColumn("is_weekend", F.dayofweek("full_date").isin(1, 7))
    .select("date_key", "full_date", "day", "month", "month_name",
            "quarter", "year", "day_of_week", "is_weekend")
)

# This is a plain batch write, NOT a stream — dim_date isn't
# "incrementally arriving" data, it's a static calendar we
# generate once. No readStream/writeStream needed here at all.
(dim_date.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("azuresalesdatabricks.gold.dim_date"))

# 3. create the dim_category table

In [0]:
%sql
CREATE TABLE IF NOT EXISTS azuresalesdatabricks.gold.dim_category (
  category_id      STRING,
  category_name    STRING,
  subcategory_name STRING
) USING DELTA;

# 4. create the dim_customer table

In [0]:
%sql
CREATE TABLE IF NOT EXISTS azuresalesdatabricks.gold.dim_customer (
  customer_key STRING, 
  customer_id STRING, 
  first_name STRING, 
  last_name STRING,
  email STRING, 
  phone STRING, 
  address STRING, 
  city STRING, 
  state STRING,
  zip_code STRING, 
  country STRING, 
  segment STRING,
  effective_start_date DATE, 
  effective_end_date DATE, 
  is_current BOOLEAN
) USING DELTA;

# 5. create the dim_product table

In [0]:
%sql
CREATE TABLE IF NOT EXISTS azuresalesdatabricks.gold.dim_product (
  product_key           STRING,
  product_id            STRING,
  product_name          STRING,
  category_id           STRING,
  brand                 STRING,
  unit_cost             DOUBLE,
  unit_price            DOUBLE,
  effective_start_date  DATE,
  effective_end_date    DATE,
  is_current            BOOLEAN
) USING DELTA;

# 6. create the dim_store table

In [0]:
%sql
CREATE TABLE IF NOT EXISTS azuresalesdatabricks.gold.dim_store (
  store_key             STRING,
  store_id              STRING,
  store_name            STRING,
  region                STRING,
  city                  STRING,
  state                 STRING,
  store_type            STRING,
  effective_start_date  DATE,
  effective_end_date    DATE,
  is_current            BOOLEAN
) USING DELTA;

# 7. create the dim_sales_rep table

In [0]:
%sql
CREATE TABLE IF NOT EXISTS azuresalesdatabricks.gold.dim_sales_rep (
  rep_key               STRING,
  rep_id                STRING,
  rep_name              STRING,
  email                 STRING,
  store_key             STRING,
  region                STRING,
  effective_start_date  DATE,
  effective_end_date    DATE,
  is_current            BOOLEAN
) USING DELTA;

# 8. create the fact_sales

In [0]:
%sql
CREATE TABLE IF NOT EXISTS azuresalesdatabricks.gold.fact_sales (
  order_item_id  STRING,
  order_id       STRING,
  customer_key   STRING,
  product_key    STRING,
  store_key      STRING,
  rep_key        STRING,
  date_key       STRING,
  quantity       INT,
  unit_price     DOUBLE,
  discount_pct   INT,
  line_total     DOUBLE,
  order_status   STRING
) USING DELTA;

# 9. create fact_returns

In [0]:
%sql
CREATE TABLE IF NOT EXISTS azuresalesdatabricks.gold.fact_returns (
  return_id       STRING,
  order_item_id   STRING,
  date_key        STRING,
  return_reason   STRING,
  refund_amount   DOUBLE
) USING DELTA;

# Change effective_start_date of bronze tables

In [0]:
%sql
UPDATE azuresalesdatabricks.silver.products SET effective_start_date = DATE('1900-01-01') WHERE is_current = true;
UPDATE azuresalesdatabricks.silver.stores SET effective_start_date = DATE('1900-01-01') WHERE is_current = true;
UPDATE azuresalesdatabricks.silver.sales_reps SET effective_start_date = DATE('1900-01-01') WHERE is_current = true;

In [0]:
%sql
select * from azuresalesdatabricks.silver.sales_reps;

# Drop specific gold tables

In [0]:
%sql
DROP TABLE IF EXISTS azuresalesdatabricks.gold.dim_product;
DROP TABLE IF EXISTS azuresalesdatabricks.gold.dim_store;
DROP TABLE IF EXISTS azuresalesdatabricks.gold.dim_sales_rep;
DROP TABLE IF EXISTS azuresalesdatabricks.gold.fact_sales;

# Remove checkpoints

In [0]:
for entity in ["dim_product", "dim_store", "dim_sales_rep", "fact_sales"]:
    dbutils.fs.rm(f"abfss://gold@azuresales.dfs.core.windows.net/_checkpoints/{entity}/", recurse=True)